# DBSCAN: estimación de `eps` con gráfico k-distance

En este notebook vamos a usar el **gráfico de distancias al vecino más cercano** (*k-distance graph*) para estimar el valor de `eps` en DBSCAN.

## 1. Importación de librerías

Usaremos `scikit-learn` para generar datos, escalar variables, calcular vecinos cercanos y aplicar DBSCAN.

In [11]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN

## 2. Dataset de ejemplo

Para que el notebook sea reutilizable, generamos un conjunto de datos sintético con tres grupos y algunos puntos atípicos.

Si quieres usar tu propio dataset, sustituye esta parte por la carga de tus datos y deja en `X` las variables numéricas que quieras utilizar para DBSCAN.

In [ ]:
# Datos agrupados artificiales
X, y_real = make_blobs(
    n_samples=300,
    centers=3,
    cluster_std=0.65,
    random_state=42
)

# Añadimos algunos posibles outliers manualmente
outliers = np.array([
    [6, 6],
    [-6, -5],
    [7, -4],
    [-5, 5],
    [0, 7]
])

X = np.vstack([X, outliers])

df = pd.DataFrame(X, columns=["variable_1", "variable_2"])
df.head()

## 3. Visualización inicial

Antes de aplicar DBSCAN, observamos la distribución general de los datos.

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df["variable_1"], df["variable_2"], s=30)
plt.xlabel("Variable 1")
plt.ylabel("Variable 2")
plt.title("Datos originales")
plt.grid(True)
plt.show()

## 4. Escalado de variables

DBSCAN se basa en distancias. Por eso, si las variables están en escalas distintas, una variable puede pesar mucho más que otra.

Por seguridad, escalamos los datos con `StandardScaler`.

In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[["variable_1", "variable_2"]])

## 5. Justificación de `min_samples`

En DBSCAN, `min_samples` indica cuántos puntos debe haber, como mínimo, dentro del radio `eps` para considerar que un punto pertenece a una zona densa.

Una regla práctica habitual es:

> `min_samples = número de dimensiones + 1`

En este caso usamos dos variables, por lo que el valor mínimo razonable sería 3. Sin embargo, para hacer el algoritmo algo más robusto frente al ruido, utilizaremos:

> `min_samples = 5`

Cuanto mayor sea `min_samples`, más exigente será DBSCAN para formar clusters y más puntos podrán aparecer como ruido.

In [5]:
min_samples = 5

## 6. Gráfico k-distance para estimar `eps`

Para construir el gráfico k-distance:

1. Calculamos los `min_samples` vecinos más cercanos de cada punto.
2. Nos quedamos con la distancia al **k-ésimo vecino**, donde `k = min_samples`.
3. Ordenamos esas distancias de menor a mayor.
4. Buscamos visualmente el **codo** de la gráfica.

Ese codo nos da una estimación razonable de `eps`.

In [ ]:
neighbors = NearestNeighbors(n_neighbors=min_samples)
neighbors_fit = neighbors.fit(X_scaled)

distances, indices = neighbors_fit.kneighbors(X_scaled)

# Distancia al k-ésimo vecino más cercano
k_distances = distances[:, min_samples - 1]

# Ordenamos las distancias
k_distances_sorted = np.sort(k_distances)

plt.figure(figsize=(8, 5))
plt.plot(k_distances_sorted)
plt.xlabel("Puntos ordenados")
plt.ylabel(f"Distancia al {min_samples}º vecino más cercano")
plt.title("Gráfico k-distance para estimar eps")
plt.grid(True)
plt.show()

## 7. Elección de `eps`

Observando el gráfico anterior, debemos localizar el punto en el que la curva empieza a crecer de forma más brusca.

Ese punto se conoce como **codo**.

En este ejemplo, elegimos un valor aproximado. Puedes modificarlo después de observar tu propia gráfica.

In [ ]:
eps = 0.35
print("Valor elegido para eps:", eps)

## 8. Aplicación de DBSCAN

Aplicamos DBSCAN con los valores seleccionados:

- `eps`: radio máximo de vecindad.
- `min_samples`: número mínimo de puntos para formar una zona densa.

DBSCAN asigna la etiqueta `-1` a los puntos que considera **ruido**.

In [ ]:
dbscan = DBSCAN(eps=eps, min_samples=min_samples)
clusters = dbscan.fit_predict(X_scaled)

df["cluster_dbscan"] = clusters

df["cluster_dbscan"].value_counts().sort_index()

## 9. Visualización de clusters y puntos de ruido

En la siguiente gráfica:

- Los puntos con etiqueta `-1` son ruido.
- El resto de valores corresponden a clusters encontrados por DBSCAN.

In [ ]:
plt.figure(figsize=(8, 6))

# Puntos que no son ruido
mask_clusters = df["cluster_dbscan"] != -1
plt.scatter(
    df.loc[mask_clusters, "variable_1"],
    df.loc[mask_clusters, "variable_2"],
    c=df.loc[mask_clusters, "cluster_dbscan"],
    s=35,
    label="Clusters"
)

# Puntos de ruido
mask_ruido = df["cluster_dbscan"] == -1
plt.scatter(
    df.loc[mask_ruido, "variable_1"],
    df.loc[mask_ruido, "variable_2"],
    marker="x",
    s=100,
    label="Ruido / posibles outliers"
)

plt.xlabel("Variable 1")
plt.ylabel("Variable 2")
plt.title("Clusters detectados por DBSCAN")
plt.legend()
plt.grid(True)
plt.show()

## 10. Relación entre ruido y outliers

Los puntos etiquetados como `-1` por DBSCAN suelen corresponderse con observaciones alejadas de zonas densas.

Esto no significa automáticamente que sean errores, pero sí que son puntos con un comportamiento diferente al de la mayoría.

Por tanto, conviene compararlos con los outliers detectados en el preprocesamiento.

In [ ]:
# Mostramos los puntos considerados ruido por DBSCAN
puntos_ruido = df[df["cluster_dbscan"] == -1]
puntos_ruido